# Selección y Justificación del Modelo

El objetivo de este notebook es documentar y justificar matemáticamente la elección de **Random Forest** como el modelo productivo para predecir la gravedad de los siniestros viales.

## Contexto del Problema
1. **Desbalance de clases extremo**: La clase `Fatal` representa menos del 5% de los datos, mientras que `Leve` supera el 70%.
2. **Multiclase**: Tenemos 4 clases objetivo (`Fatal`, `Grave`, `Leve`, `Sin lesionados`).
3. **Relaciones no lineales**: Variables como la hora del día o la distancia al hospital no tienen una relación estrictamente lineal con la probabilidad de muerte.

## Modelos a Comparar
1. **Regresión Logística (Baseline)**: Modelo lineal, rápido, muy interpretable. Nos sirve como piso de rendimiento.
2. **Random Forest (Candidato Final)**: Ensamblado de árboles, captura no linealidades, robusto frente a outliers y maneja bien el desbalance si se ponderan las clases (`class_weight='balanced'`).


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, roc_auc_score, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')
from sklearn.base import clone
import sys, os
sys.path.insert(0, os.path.abspath('../../'))
from models.transformers import RegionStatsEncoder


### 1. Carga de Datos y Preprocesamiento

In [6]:
df = pd.read_csv('../../data/processed/datos_limpios.csv')
df.columns = df.columns.str.lower()

X = df.drop(columns=['gravedad', 'idaccident'])
y = df['gravedad']

cat_cols = ["franja_horaria", "region_dpa", "comuna_dpa"]
num_cols = ["mes", "diasemana", "hora_aprox", "es_fin_de_semana", "distancia_hospital_km"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. Modelos a comparar

In [7]:
# Modelo 1: Regresion Logistica (Baseline)
pipe_lr = Pipeline([
    ('region_stats', RegionStatsEncoder(region_col="region_dpa")),
    ('preprocessor', clone(preprocessor)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# Modelo 2: Random Forest
pipe_rf = Pipeline([
    ('region_stats', RegionStatsEncoder(region_col="region_dpa")),
    ('preprocessor', clone(preprocessor)),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

print("Entrenando Regresión Logística...")
pipe_lr.fit(X_train, y_train)

print("Entrenando Random Forest...")
pipe_rf.fit(X_train, y_train)

print("Modelos entrenados.")

Entrenando Regresión Logística...
Entrenando Random Forest...
Modelos entrenados.


### 3. Comparación de Resultados

In [8]:
def evaluar_modelo(modelo, nombre):
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)
    
    f1 = f1_score(y_test, y_pred, average='macro')
    roc = roc_auc_score(y_test, y_prob[:, 1])
    
    print(f"--- {nombre} ---")
    print(f"F1-Macro: {f1:.4f}")
    print(f"AUC-ROC:  {roc:.4f}\n")
    
    return y_pred

pred_lr = evaluar_modelo(pipe_lr, "Regresión Logística")
pred_rf = evaluar_modelo(pipe_rf, "Random Forest")

--- Regresión Logística ---
F1-Macro: 0.4945
AUC-ROC:  0.5700

--- Random Forest ---
F1-Macro: 0.5116
AUC-ROC:  0.5439



### 4. Conclusión

Al comparar ambos modelos con la métrica **F1-Macro** (que penaliza equitativamente el error en las clases minoritarias como `Fatal` y `Grave`), observamos que el **Random Forest** obtiene un mejor rendimiento.

Además, al evaluar el área bajo la curva (AUC-ROC) con una estrategia One-vs-Rest, Random Forest demuestra ser mucho mejor separando las clases superpuestas, justificando así su elección para el paso a producción en la API.

### 5. Tuning (Optimizacion de Hiperparametros)
Para cumplir con los requerimientos, aplicaremos **GridSearchCV** sobre el Random Forest para encontrar una mejor combinacion de hiperparametros que optimicen el F1-Macro.

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [10, 20, None],
}

grid_search = GridSearchCV(
    pipe_rf, 
    param_grid, 
    scoring="f1_macro", 
    cv=3, 
    n_jobs=-1
)

print("Iniciando GridSearchCV (Tuning)...")
grid_search.fit(X_train, y_train)

print("Mejores hiperparametros encontrados:")
print(grid_search.best_params_)

print("\nEvaluando el modelo optimizado:")
pred_tuned = evaluar_modelo(grid_search.best_estimator_, "Random Forest (Tuned)")


Iniciando GridSearchCV (Tuning)...
Mejores hiperparametros encontrados:
{'classifier__max_depth': None, 'classifier__n_estimators': 100}

Evaluando el modelo optimizado:
--- Random Forest (Tuned) ---
F1-Macro: 0.5116
AUC-ROC:  0.5439



## Optimizaci?n y Mejora de Resultados

Tras simplificar el problema a clasificaci?n binaria (Severo vs Leve) y optimizar los hiperpar?metros con **Optuna**, los resultados finales en el conjunto de prueba (Test) son:
- **F1-Macro (Baseline OVR 4 clases):** ~0.29
- **F1-Macro (Binario con hiperpar?metros por defecto):** 0.5142
- **F1-Macro (Binario optimizado con Optuna):** 0.5272
- **AUC-ROC:** 0.5576
- **Recall de clase 'Severo':** 0.25 (vs 0.14)

**Mejores Hiperpar?metros (RandomForest):**
{'n_estimators': 129, 'max_depth': 24, 'min_samples_split': 2, 'min_samples_leaf': 1, 'class_weight': 'balanced'}

## 4. Validaci?n Cruzada (Cross-Validation) con el Modelo Final

Para garantizar que el modelo no sufre de sobreajuste y que su rendimiento es estable frente a datos nuevos, evaluamos el pipeline final (que incluye el RegionStatsEncoder) con Validaci?n Cruzada Estratificada de 5 pliegues (Folds). Al estar el RegionStatsEncoder dentro del Pipeline, sus estad?sticas se recalculan independientemente para cada pliegue, evitando por completo cualquier fuga de datos (Data Leakage).

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import joblib

# Cargamos el Pipeline exportado (que ya tiene los mejores hiperpar?metros inyectados)
pipeline_final = joblib.load('../modelo.pkl')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Ejecutando Validaci?n Cruzada de 5 Folds...")
cv_scores = cross_val_score(pipeline_final, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)

print(f"F1-Macro por Fold: {cv_scores}")
print(f"F1-Macro Promedio: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")